In [2]:
# ==========================================
# 1. Install Official Google TF-Keras
# ==========================================
!pip install -q tf_keras kagglehub

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import tensorflow as tf
import tf_keras as keras
import tensorflow_hub as hub
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from shutil import copyfile
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report

from tf_keras.preprocessing.image import ImageDataGenerator
from tf_keras.layers import Dense, Dropout, Input, BatchNormalization
from tf_keras.models import Model
from tf_keras.callbacks import EarlyStopping, ModelCheckpoint

print("\n🚀 Starting All-in-One Safe ViT Transfer Learning Pipeline...\n")

# ==========================================
# STEP 1: KAGGLE AUTHENTICATION & DOWNLOAD
# ==========================================
print("🔑 Kaggle Authentication Setup...")
kaggle_username = input("Enter your Kaggle Username: ")
kaggle_key = input("Enter your Kaggle API Key: ")

os.environ['KAGGLE_USERNAME'] = kaggle_username
os.environ['KAGGLE_KEY'] = kaggle_key

zip_name = 'skin-cancer-mnist-ham10000.zip'
raw_extract_path = '/content/raw_kaggle_data'

if not os.path.exists(zip_name) and not os.path.exists(raw_extract_path):
    print("\n⬇️ Downloading dataset directly from Kaggle... ⏳")
    !kaggle datasets download -d kmader/skin-cancer-mnist-ham10000
    print("✅ Download Complete!")

if not os.path.exists(raw_extract_path):
    print("📦 Extracting raw Kaggle files... ⏳")
    import zipfile
    with zipfile.ZipFile(zip_name, 'r') as zip_ref:
        zip_ref.extractall(raw_extract_path)
    print("✅ Raw Extraction Complete!")

# ==========================================
# STEP 2: PATIENT-LEVEL SPLIT
# ==========================================
base_dir = '/content/skin_disease_data'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')

runs_dir = '/content/runs/vit_exp'
weights_dir = os.path.join(runs_dir, 'weights')
os.makedirs(weights_dir, exist_ok=True)

img_dirs = [os.path.join(raw_extract_path, 'HAM10000_images_part_1'),
            os.path.join(raw_extract_path, 'HAM10000_images_part_2')]

if not os.path.exists(train_dir):
    print("\n🛡️ Performing Patient-Level Split based on 'lesion_id'...")
    metadata_path = os.path.join(raw_extract_path, 'HAM10000_metadata.csv')
    df = pd.read_csv(metadata_path)

    unique_lesions = df.groupby('lesion_id').first().reset_index()
    train_lesions, val_lesions = train_test_split(
        unique_lesions['lesion_id'], test_size=0.2, random_state=42, stratify=unique_lesions['dx']
    )

    train_lesions_set = set(train_lesions)
    val_lesions_set = set(val_lesions)

    for idx, row in df.iterrows():
        img_id = row['image_id']
        lesion_id = row['lesion_id']
        label = row['dx']

        target_sub = train_dir if lesion_id in train_lesions_set else val_dir
        target_class_dir = os.path.join(target_sub, label)
        os.makedirs(target_class_dir, exist_ok=True)

        for d in img_dirs:
            src_img_path = os.path.join(d, f"{img_id}.jpg")
            if os.path.exists(src_img_path):
                copyfile(src_img_path, os.path.join(target_class_dir, f"{img_id}.jpg"))
                break
    print("✅ Dataset successfully split and balanced!")
else:
    print("✅ Directory structure already exists. Skipping image copy.")

# ==========================================
# STEP 3: DATA GENERATORS
# ==========================================
print("\nConfiguring Data Augmentation (Batch Size: 16)...")
train_datagen = ImageDataGenerator(rescale=1./255, rotation_range=20, width_shift_range=0.1, height_shift_range=0.1, horizontal_flip=True)
val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(train_dir, target_size=(224, 224), batch_size=16, class_mode='categorical')
val_generator = val_datagen.flow_from_directory(val_dir, target_size=(224, 224), batch_size=16, class_mode='categorical', shuffle=False)

class_names = list(train_generator.class_indices.keys())
class_weights_array = compute_class_weight(class_weight='balanced', classes=np.unique(train_generator.classes), y=train_generator.classes)
class_weights = dict(enumerate(class_weights_array))

# ==========================================
# STEP 4: TF-HUB ViT (FROZEN TRANSFER LEARNING)
# ==========================================
print("\n🏗️ Loading Official Google ViT-B/16 (Frozen for Stability)...")
vit_model_path = kagglehub.model_download('spsayakpaul/vision-transformer/tensorFlow2/vit-b16-fe')

# STRICTLY FROZEN: Taake GPU OOM Crash na ho
base_model = hub.KerasLayer(vit_model_path, trainable=False, name='vit_layer')

inputs = Input(shape=(224, 224, 3))
x = base_model(inputs)
x = BatchNormalization()(x) # Faster convergence
x = Dropout(0.5)(x)
x = Dense(256, activation='relu')(x) # Extra layer for better learning
x = Dropout(0.3)(x)
outputs = Dense(7, activation='softmax')(x)

model = Model(inputs=inputs, outputs=outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')])

best_path = os.path.join(weights_dir, 'vit_best_model.keras')
callbacks = [EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
             ModelCheckpoint(best_path, save_best_only=True, monitor='val_accuracy')]

# ==========================================
# STEP 5: TRAINING (ONE PHASE ONLY)
# ==========================================
print("\n🔥 Training Custom Classification Head (Up to 20 Epochs)...")
history = model.fit(train_generator, epochs=20, validation_data=val_generator, class_weight=class_weights, callbacks=callbacks)

# ==========================================
# STEP 6: EVALUATION & PLOTTING
# ==========================================
print("\n📊 Generating ViT Evaluation Metrics and Plots...")

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy', color='blue', linewidth=1.5)
plt.plot(epochs_range, val_acc, label='Validation Accuracy', color='green', linewidth=1.5)
plt.title('ViT Model Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss', color='red', linewidth=1.5)
plt.plot(epochs_range, val_loss, label='Validation Loss', color='orange', linewidth=1.5)
plt.title('ViT Model Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.savefig(os.path.join(runs_dir, 'metrics_curve.png'), dpi=300)
plt.close()

print("Generating Confusion Matrix...")
Y_pred = model.predict(val_generator)
y_pred = np.argmax(Y_pred, axis=1)
y_true = val_generator.classes

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Vision Transformer (ViT)')
plt.ylabel('True Class')
plt.xlabel('Predicted Class')
plt.savefig(os.path.join(runs_dir, 'confusion_matrix.png'), dpi=300)
plt.close()

report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
df_report = pd.DataFrame(report).transpose()
df_report.to_csv(os.path.join(runs_dir, 'classification_report.csv'))

print(f"\n🎉 ALL ViT ASSETS SAVED SUCCESSFULLY IN: {runs_dir}")

import shutil
shutil.make_archive('/content/vit_runs_complete', 'zip', '/content/runs/vit_exp')
print("📦 Directory zipped successfully. Ready for download!")

from google.colab import files
files.download('/content/vit_runs_complete.zip')


🚀 Starting All-in-One Safe ViT Transfer Learning Pipeline...

🔑 Kaggle Authentication Setup...
Enter your Kaggle Username: haroonbaloshi868
Enter your Kaggle API Key: 203ee5d839ff48123961759616aa31b9

⬇️ Downloading dataset directly from Kaggle... ⏳
Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0
100% 5.20G/5.20G [00:43<00:00, 127MB/s] 

✅ Download Complete!
📦 Extracting raw Kaggle files... ⏳
✅ Raw Extraction Complete!

🛡️ Performing Patient-Level Split based on 'lesion_id'...
✅ Dataset successfully split and balanced!

Configuring Data Augmentation (Batch Size: 16)...
Found 8017 images belonging to 7 classes.
Found 1998 images belonging to 7 classes.

🏗️ Loading Official Google ViT-B/16 (Frozen for Stability)...



100%|██████████| 11.4k/11.4k [00:00<00:00, 9.09MB/s]



  0%|          | 0.00/327M [00:00<?, ?B/s]



  0%|          | 0.00/3.67M [00:00<?, ?B/s]
  0%|          | 1.00M/327M [00:00<02:08, 2.66MB/s]

 27%|██▋       | 1.00M/3.67M [00:00<00:01, 2.22MB/s]
  1%|          | 3.00M/327M [00:00<00:46, 7.29MB/s]
  3%|▎         | 9.00M/327M [00:00<00:15, 21.0MB/s]

100%|██████████| 3.67M/3.67M [00:00<00:00, 6.15MB/s]

  4%|▍         | 13.0M/327M [00:00<00:12, 26.4MB/s]
  5%|▌         | 18.0M/327M [00:00<00:09, 33.3MB/s]
  8%|▊         | 25.0M/327M [00:00<00:07, 43.7MB/s]
  9%|▉         | 30.0M/327M [00:01<00:06, 46.0MB/s]
 11%|█▏        | 37.0M/327M [00:01<00:06, 49.4MB/s]
 13%|█▎        | 43.0M/327M [00:01<00:08, 36.5MB/s]
 15%|█▌        | 50.0M/327M [00:01<00:06, 41.8MB/s]
 17%|█▋        | 57.0M/327M [00:01<00:06, 46.0MB/s]
 20%|█▉        | 64.0M/327M [00:01<00:05, 49.0MB/s]
 22%|██▏       | 71.0M/327M [00:01<00:05, 51.2MB/s]
 24%|██▍       | 78.0M/327M [00:02<00:04, 52.9MB/s]
 26%|██▌       | 85.0M/327M [00:02<00:04, 54.1MB/s]
 28%|██▊       | 91.0M/327M [00:02<00:04, 52.6MB/s]
 30%|██▉     


🔥 Training Custom Classification Head (Up to 20 Epochs)...
Epoch 1/20
502/502 [==============================] - 231s 432ms/step - loss: 1.8306 - accuracy: 0.5080 - precision: 0.5621 - recall: 0.4579 - val_loss: 1.5016 - val_accuracy: 0.5400 - val_precision: 0.5762 - val_recall: 0.5015
Epoch 2/20
502/502 [==============================] - 210s 417ms/step - loss: 1.2801 - accuracy: 0.5825 - precision: 0.6417 - recall: 0.5321 - val_loss: 1.1858 - val_accuracy: 0.5896 - val_precision: 0.6285 - val_recall: 0.5460
Epoch 3/20
502/502 [==============================] - 198s 394ms/step - loss: 1.1541 - accuracy: 0.5876 - precision: 0.6509 - recall: 0.5297 - val_loss: 1.3096 - val_accuracy: 0.5541 - val_precision: 0.6002 - val_recall: 0.5065
Epoch 4/20
502/502 [==============================] - 193s 384ms/step - loss: 1.0202 - accuracy: 0.6095 - precision: 0.6779 - recall: 0.5500 - val_loss: 1.2741 - val_accuracy: 0.5501 - val_precision: 0.6013 - val_recall: 0.4945
Epoch 5/20
502/502 [========

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>